In [1]:
!pip install "datasets<3.0.0" soundfile

!pip install transformers accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 9.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
!pip install resemblyzer  # lightweight speaker embedding library

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 49.5 MB/s eta 0:00:00
  Created wheel for webrtcvad: filename=webrtcvad-2.0.10-cp312-cp312-linux_x86_64.whl size=73519 sha256=b07a97ffad97a3aa568c66636a682fb036ed9f32f86464726f6a2634751e224c
  Stored in directory: /root/.cache/pip/wheels/1e/d3/95/680fa3b16848f1a58d2edaed34c496224c89a9bc63e17b3614
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26304 sha256=9a62c0e97404803e8ad10d8366c700391382fc889c6e75e395663cb0fd321c6b
  Stored in directory: /root/.cache/pip/wheels/12/98/52/2bffe242a9a487f00886e43b8ed8dac46456702e11a0d6abef
Successfully built webrtcvad typing


In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
from datasets import load_dataset

# Target Urdu directory
ds_urduspeech_stream = load_dataset(
    "humairawan/UrduSpeech",
    data_dir="Urdu",
    split="train",
    token='',
    streaming=True
)

# Take first NUM_SAMPLES with style = "BOOK"
urduspeech_samples = []
for sample in ds_urduspeech_stream:
    if sample.get("style") == "BOOK" and sample.get("gender") == "Male":  # filter book literary
        urduspeech_samples.append(sample)
        print(f"Collected samples: {len(urduspeech_samples)}")
    if len(urduspeech_samples) >= 60:
        break # first sample

Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Collected samples: 1
Collected samples: 2
Collected samples: 3
Collected samples: 4
Collected samples: 5
Collected samples: 6
Collected samples: 7
Collected samples: 8
Collected samples: 9
Collected samples: 10
Collected samples: 11
Collected samples: 12
Collected samples: 13
Collected samples: 14
Collected samples: 15
Collected samples: 16
Collected samples: 17
Collected samples: 18
Collected samples: 19
Collected samples: 20
Collected samples: 21
Collected samples: 22
Collected samples: 23
Collected samples: 24
Collected samples: 25
Collected samples: 26
Collected samples: 27
Collected samples: 28
Collected samples: 29
Collected samples: 30
Collected samples: 31
Collected samples: 32
Collected samples: 33
Collected samples: 34
Collected samples: 35
Collected samples: 36
Collected samples: 37
Collected samples: 38
Collected samples: 39
Collected samples: 40
Collected samples: 41
Collected samples: 42
Collected samples: 43
Collected samples: 44
Collected samples: 45
Collected samples: 

In [7]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(0,10):

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.01 seconds.
Sample 0: یا یہ طلسم ہے کہ اگر پھٹکری اور گندھک کو...
Sample 1: میں جان و دل سے اُسے چاہتی تھی، اُس کی ب...
Sample 2: سب آدمی اپنے اپنے عہدوں پر مستعد ہیں، با...
Waiting for 1.2 minutes...
Sample 3: تب انہوں نے فرمایا کہ مرتضیٰ رضی اللہ می...
Sample 4: تم سخاوت کا بوجھ نہیں اٹھا سکتے۔...
Sample 5: سوائے اِس بات کے زبان سے کچھ نہ نکلا، فی...
Waiting for 1.2 minutes...
Sample 6: آگے چوکی پر ڈونگے کٹورے، بمعِ تھالی، سری...
Sample 7: اور پھر ایک بار گئی، عاجز کو یوں سر بلند...
Sample 8: سچ مچ یہ تماشا ہوا جیسے چودھویں رات کے چ...
Waiting for 1.2 minutes...
Sample 9: جب یہ احوال ناامیدی کا سنا، ایسی بدحواس ...


In [8]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(10,20):

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.02 seconds.
Sample 10: غرض بہتری خاک چھانی لیکن اُس گوہر نایاب ...
Sample 11: نظر کی مجال نہ تھی جو اس کے جمال پر ٹھہر...
Waiting for 1.2 minutes...
Sample 12: بھلا ایک زخم اور بھی لگا، میں نے اپنا تی...
Sample 13: میں ابتدا سے کہتا ہوں، تا انتہا سنو!...
Sample 14: ہرچند انہوں نے میرے غائب ہونے کی کیفیت د...
Waiting for 1.2 minutes...
Sample 15: بارے خدا خدا کر کے صبح جب نزدیک ہوئی، مر...
Sample 16: یہ دونوں کی باتیں حاتم نے سنیں، مروی ہے ...
Sample 17: میرے طرف دھیان نہ کیا۔...
Waiting for 1.2 minutes...
Sample 18: موافقِ قدر و منزلت کے ہر ایک کو سر فرازی...
Sample 19: تو نے جان و مال سے میری خاطر کی اور جو ک...


In [9]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(20,30):

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.02 seconds.
Sample 20: جب نو فل کے روبرو لے گئے تو اس نے یہ پوچ...
Waiting for 1.2 minutes...
Sample 21: سن کر مسکرایا اور بولا "مناسب ہے کہ صاحب...
Sample 22: میں نے بہت منت کی اور ہاتھ جوڑ کر کہا، ت...
Sample 23: ساری رات دروازے گھروں کے بند نہ ہوتے اور...
Waiting for 1.2 minutes...
Sample 24: طاقت بدن میں مطلق نہ رہی،اپاہج ہو کر اسی...
Sample 25: حکم کرتے ہی تھوڑے دنوں میں ایسی نقب تیار...
Sample 26: میں اپنے دل کو ہر چند سنبھالتی پر اس کاف...
Waiting for 1.2 minutes...
Sample 27: اُس مرد آدمی نے کہا، یہ وہی کم بخت بد نص...
Sample 28: راہ میں جو کچھ مصیبتیں قسمت میں لکھی تھی...
Sample 29: تمام دن جیسے روزہ دار شام ہونے کا انتظار...
Waiting for 1.2 minutes...


In [10]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(30,40):

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.01 seconds.
Sample 30: غرض جس کے گھر میں اتنی دولت اور ایک لڑکا...
Sample 31: اس واسطے کہ آدمی کا دل خدا کا گھر ہے۔...
Sample 32: پچھلے پہر ڈاکا آیا، جو کچھ مال و اسباب پ...
Waiting for 1.2 minutes...
Sample 33: تونے مجھے نہال کیا، لیکن مردوں کو خدا نے...
Sample 34: غرض، قسمت کی خوبی اس ملک کی تھی، جوایسا ...
Sample 35:  آدمان باتوں سے سوائے اُس خوج کے اور دود...
Waiting for 1.2 minutes...
Sample 36: دو ہیں طبیب آ کر جمع ہوئے۔...
Sample 37: دیر نہ کیجیے، سچ ہے معشوق بن کر کچھ اچھا...
Sample 38: نبضِ قارورہ دیکھ کر بہت غور کیا۔...
Waiting for 1.2 minutes...
Sample 39: جس کی یہ قدرت اور سکت ہو، سکی حمد و ثنا ...


In [11]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(40,50):

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.02 seconds.
Sample 40: پس عبادت بھی اس روز کام نہ آئے گی۔...
Sample 41: فی الفور قلمدان آگے رکھ دیا۔...
Waiting for 1.2 minutes...
Sample 42: ہر دم اُس کی خاطر داری کرتی، آخر کو میری...
Sample 43: اندر سے گھوراکر بولے، اس وقت دروازہ کھول...
Sample 44: ندہڑک بول اٹھا کہ اب اِس طور کی زندگی کو...
Waiting for 1.2 minutes...
Sample 45: یہ کہاوت ہے۔...
Sample 46: ایک روز ایک مصاحب دانانہ کہ خوب تواریخ د...
Sample 47: ہمارے ہاتھ حاتم کا ہے؟ کوئی آوے گا اور ب...
Waiting for 1.2 minutes...
Sample 48: اس کا عذاب میرے نام لکھا جائے گا۔...
Sample 49: میں نے سارا احوال مول تول کا اور مہمانی ...


In [12]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in range(50,60):

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.02 seconds.
Sample 50: میں نے معلوم کیا کہ اس احمق نے بڑی خواہش...
Waiting for 1.2 minutes...
Sample 51: فقیر اس کے دیکھنے سے ڈر گیا۔...
Error on sample 52: list index out of range
Sample 53: جب تھوڑی سی رات باقی رہی، بادشاہ زادی مر...
Waiting for 1.2 minutes...
Sample 54: میں حیران ہوا لیکن اپنا گھر جان کر قدم ا...
Sample 55: ہر ایک سے پوچھتا پھرتا تھا کہ اس شہر میں...
Sample 56: اتنے عرصے میں یہ سب تیاری کیوں کر ہوئی؟...
Waiting for 1.2 minutes...
Sample 57: آدمی اناج کا کیڑا ہے۔...
Sample 58: یہ کہہ کر اسی بے ہوشی کے عالم میں دوپٹے ...
Sample 59: کیا جانیں یہ دیو ہیں یا غول بیابانی ہے ہ...
Waiting for 1.2 minutes...


In [14]:
from datasets import load_dataset
import google.generativeai as genai
import io, wave
import numpy as np
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
import time

# =========================
# Setup
# =========================
genai.configure(api_key="")
model = genai.GenerativeModel("gemini-2.5-flash-preview-tts")
urdu_conv = urduspeech_samples
encoder = VoiceEncoder()

scores = []
results = []

for i in [52]:

    text = urdu_conv[i]['text']
    ref_array = urdu_conv[i]['audio']['array'].astype(np.float32)
    ref_sr = urdu_conv[i]['audio']['sampling_rate']

    ref_path = f"ref_{i}.wav"
    gen_path = f"gen_{i}.wav"

    # Save reference
    sf.write(ref_path, ref_array, samplerate=ref_sr)

    # =========================
    # Gemini TTS
    # =========================
    speech_config = {
        "voice_config": {
            "prebuilt_voice_config": {
                "voice_name": "charon"
            }
        }
    }

    response = model.generate_content(
        f"Please read this text: {text}",
        generation_config={
            "response_modalities": ["AUDIO"],
            "speech_config": speech_config
        }
    )

    try:
        audio_bytes = response.candidates[0].content.parts[0].inline_data.data

        byte_io = io.BytesIO()
        with wave.open(byte_io, 'wb') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(24000)
            wav_file.writeframes(audio_bytes)

        byte_io.seek(0)

        with open(gen_path, "wb") as f:
            f.write(byte_io.read())

    except Exception as e:
        print(f"Error on sample {i}: {e}")
        continue
    print(f"Sample {i}: {text[:40]}...")
    # =========================
    # Delay after every 3 iterations
    # =========================
    if (i + 1) % 3 == 0:
        print("Waiting for 1.2 minutes...")
        time.sleep(72)

Loaded the voice encoder model on cpu in 0.01 seconds.
Sample 52: چندے بیکاری گزری۔...


In [15]:
# ============================================
# PHASE 2: Package audio files for MUSHRA
# ============================================
import os
import zipfile

# Create folder structure webMUSHRA expects
os.makedirs("mushra_audio/reference", exist_ok=True)
os.makedirs("mushra_audio/generated_gemini_tts", exist_ok=True)

for i in range(60):
    # Copy reference
    ref_src = f"ref_{i}.wav"
    gen_src = f"gen_{i}.wav"

    # Read files
    ref_data, ref_sr = sf.read(ref_src)
    gen_data, gen_sr = sf.read(gen_src)

    # Save into structured folders
    sf.write(f"mushra_audio/reference/sample_{i}.wav", ref_data, ref_sr)
    sf.write(f"mushra_audio/generated_gemini_tts/sample_{i}.wav", gen_data, gen_sr)


# Zip everything for download
with zipfile.ZipFile("60_mushra_book_gemini_audio.zip", "w") as zf:
    for root, dirs, files in os.walk("mushra_audio"):
        for file in files:
            zf.write(os.path.join(root, file))

print("60_mushra_audio.zip ready for download!")

# Download in Colab
from google.colab import files
files.download("60_mushra_book_gemini_audio.zip")

60_mushra_audio.zip ready for download!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>